In [1]:
import scCube
from scCube import scCube
from scCube.visualization import *
from scCube.utils import *
from matplotlib.pyplot import rc_context
import pandas as pd
import scanpy as sc
import numpy as np
import warnings
import time
warnings.filterwarnings("ignore")

from scipy import io

In [2]:
model = scCube()

In [3]:
sc_data = pd.read_csv('../../data/reference_count.csv', index_col=0)
sc_meta = pd.read_csv('../../data/reference_metadata.csv', index_col=0)
sc_meta.index = sc_meta['cell_id'].astype(str)
shared_index = sc_data.columns.intersection(sc_meta.index)
sc_data = sc_data[shared_index]
sc_meta = sc_meta.loc[shared_index]
sc_meta['Cell'] = sc_meta.index

In [4]:
sc_adata = model.pre_process(sc_data=sc_data, 
                             sc_meta=sc_meta,
                             is_normalized=False)

the input is count matrix, normalizing it firstly...


In [5]:
generate_sc_meta, generate_sc_data = model.train_vae_and_generate_cell(
    sc_adata=sc_adata,
    celltype_key='Cluster',
    cell_key='Cell',
    target_num=dict(sc_meta.Cluster.value_counts()), # target number of cells to generate, if `target_num=None`, generate cells by the proportion of cell types of the input data
    batch_size=512,
    epoch_num=20,
    lr=0.0001,
    hidden_size=128,
    save_model=False,
    used_device='cpu',)

generating by the targeted proportion of cell types...
begin vae training...


Train Epoch: 19: 100%|██████████| 20/20 [00:02<00:00,  7.98it/s, loss=10.5565, min_loss=10.5565]


vae training done!


Generate Epoch: 0: 100%|██████████| 2032/2032 [00:00<00:00, 24882.49it/s]

generated done!
data have been prepared!


In [12]:
generate_sc_data_new, generate_sc_meta_new = model.generate_pattern_random(
    generate_sc_data=generate_sc_data,
    generate_sc_meta=generate_sc_meta,
    set_seed=True,
    seed=12345,
    spatial_cell_type=None,
    spatial_dim=2,
    spatial_size=50,
    delta=10,
    lamda=0.75,)

generating spatial coordinates of single cells...
generating spatial patterns of totally 9 cell types...


In [ ]:
generate_sc_meta_new.to_csv('sccube_metadata.csv')
generate_sc_data_new.to_csv('sccube_count.csv')